# 📊 DataSense – Exploratory Data Analysis
Heart Disease Dataset · Griffith College Dublin

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')
from src.data_loader import load_data, preprocess_data

sns.set_theme(style='darkgrid')
df = load_data()
df.head()

In [ ]:
print(f'Shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()}')
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
for ax, col in zip(axes.flatten(), num_cols):
    df[col].hist(ax=ax, bins=25, color='#f5b800', edgecolor='#0d0f14')
    ax.set_title(col)
plt.suptitle('Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
from src.model import build_ensemble
from sklearn.metrics import classification_report
import shap

X_train, X_test, y_train, y_test, feature_names, preprocessor = preprocess_data(df)
model = build_ensemble()
model.fit(X_train, y_train)
preds = model.predict(X_test)
print(classification_report(y_test, preds, target_names=['No Disease', 'Disease']))

In [ ]:
# SHAP analysis
xgb = model.estimators_[0][1]
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
if isinstance(shap_values, list):
    shap_values = shap_values[1]
shap.summary_plot(shap_values, X_test, feature_names=feature_names)